# 🔹 Notebook 1 — Neo4j Graph Queries

**Obiettivo**: Testare tutte le query Cypher del libro sul Knowledge Graph.

**Prerequisiti**: Stack Docker attivo (`docker compose --profile dev up -d`)

| Servizio | URL | Credenziali |
|----------|-----|-------------|
| Neo4j Browser | http://localhost:7474 | neo4j / yourpassword |
| Redis Insight | http://localhost:5540 | — |


## Setup: connessione a Neo4j

In [ ]:
import os
from neo4j import GraphDatabase

NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://localhost:7687')
NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', 'yourpassword')

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def run_cypher(query, params=None):
    """Esegue una query Cypher e ritorna i risultati come lista di dict."""
    with driver.session() as session:
        result = session.run(query, params or {})
        return [record.data() for record in result]

# Test connessione
print('Connessione:', run_cypher('RETURN 1 AS ok'))

## 1.1 — Seed: popola il grafo con dati di esempio

In [ ]:
# ── Crea nodi Person ──
seed_queries = [
    # Persone
    "MERGE (a:Person {name: 'Alice', age: 30, email: 'alice@techcorp.com'})",
    "MERGE (b:Person {name: 'Bob', age: 28, email: 'bob@techcorp.com'})",
    "MERGE (c:Person {name: 'Carol', age: 35, email: 'carol@newtech.io'})",
    "MERGE (d:Person {name: 'Dave', age: 42, email: 'dave@finserv.com'})",
    "MERGE (e:Person {name: 'Eva', age: 29, email: 'eva@techcorp.com'})",
    # Aziende
    "MERGE (tc:Company {name: 'TechCorp', industry: 'Software', location: 'San Francisco'})",
    "MERGE (nt:Company {name: 'NewTech', industry: 'AI', location: 'Milano'})",
    "MERGE (fs:Company {name: 'FinServ', industry: 'Finance', location: 'London'})",
    # Tecnologie
    "MERGE (py:Technology {name: 'Python'})",
    "MERGE (neo:Technology {name: 'Neo4j'})",
    "MERGE (k8s:Technology {name: 'Kubernetes'})",
    "MERGE (redis:Technology {name: 'Redis'})",
    # Prodotti
    "MERGE (p1:Product {name: 'SlimWash Pro', sku: 'SW-001', price: 499.0})",
    "MERGE (p2:Product {name: 'SmartDry X', sku: 'SD-002', price: 699.0})",
    # Warranty
    "MERGE (w1:WarrantyPolicy {id: 'WP-001', duration: '24 mesi', coverage: 'difetti fabbricazione'})",
    # Relazioni
    """MATCH (a:Person {name:'Alice'}), (tc:Company {name:'TechCorp'})
       MERGE (a)-[:WORKS_AT {since: 2020, role: 'Senior Engineer'}]->(tc)""",
    """MATCH (b:Person {name:'Bob'}), (tc:Company {name:'TechCorp'})
       MERGE (b)-[:WORKS_AT {since: 2019, role: 'Data Engineer'}]->(tc)""",
    """MATCH (e:Person {name:'Eva'}), (tc:Company {name:'TechCorp'})
       MERGE (e)-[:WORKS_AT {since: 2021, role: 'ML Engineer'}]->(tc)""",
    """MATCH (c:Person {name:'Carol'}), (nt:Company {name:'NewTech'})
       MERGE (c)-[:WORKS_AT {since: 2022, role: 'CTO'}]->(nt)""",
    """MATCH (d:Person {name:'Dave'}), (fs:Company {name:'FinServ'})
       MERGE (d)-[:WORKS_AT {since: 2018, role: 'VP Engineering'}]->(fs)""",
    # Knows
    "MATCH (a:Person {name:'Alice'}),(b:Person {name:'Bob'}) MERGE (a)-[:KNOWS {since:2019}]->(b)",
    "MATCH (a:Person {name:'Alice'}),(c:Person {name:'Carol'}) MERGE (a)-[:KNOWS {since:2021}]->(c)",
    "MATCH (b:Person {name:'Bob'}),(d:Person {name:'Dave'}) MERGE (b)-[:KNOWS {since:2020}]->(d)",
    "MATCH (c:Person {name:'Carol'}),(d:Person {name:'Dave'}) MERGE (c)-[:KNOWS {since:2022}]->(d)",
    "MATCH (e:Person {name:'Eva'}),(a:Person {name:'Alice'}) MERGE (e)-[:KNOWS {since:2021}]->(a)",
    # Competenze
    "MATCH (a:Person {name:'Alice'}),(py:Technology {name:'Python'}) MERGE (a)-[:SKILLED_IN {level:'expert'}]->(py)",
    "MATCH (a:Person {name:'Alice'}),(neo:Technology {name:'Neo4j'}) MERGE (a)-[:SKILLED_IN {level:'advanced'}]->(neo)",
    "MATCH (b:Person {name:'Bob'}),(py:Technology {name:'Python'}) MERGE (b)-[:SKILLED_IN {level:'expert'}]->(py)",
    "MATCH (b:Person {name:'Bob'}),(redis:Technology {name:'Redis'}) MERGE (b)-[:SKILLED_IN {level:'advanced'}]->(redis)",
    "MATCH (e:Person {name:'Eva'}),(k8s:Technology {name:'Kubernetes'}) MERGE (e)-[:SKILLED_IN {level:'expert'}]->(k8s)",
    # Prodotti e garanzia
    "MATCH (p:Product {name:'SlimWash Pro'}),(w:WarrantyPolicy {id:'WP-001'}) MERGE (p)-[:HAS_WARRANTY]->(w)",
    "MATCH (p:Product {name:'SlimWash Pro'}),(p2:Product {name:'SmartDry X'}) MERGE (p)-[:COMPATIBLE_WITH]->(p2)",
    # Aziende e tecnologie
    "MATCH (tc:Company {name:'TechCorp'}),(py:Technology {name:'Python'}) MERGE (tc)-[:USES]->(py)",
    "MATCH (tc:Company {name:'TechCorp'}),(neo:Technology {name:'Neo4j'}) MERGE (tc)-[:USES]->(neo)",
    "MATCH (nt:Company {name:'NewTech'}),(py:Technology {name:'Python'}) MERGE (nt)-[:USES]->(py)",
]

for q in seed_queries:
    run_cypher(q)

# Verifica
counts = run_cypher('MATCH (n) RETURN labels(n)[0] AS tipo, count(n) AS conteggio ORDER BY conteggio DESC')
for c in counts:
    print(f"  {c['tipo']}: {c['conteggio']}")
print('\nSeed completato!')

## 1.2 — Ricerca Semantica: trovare per significato, non per keyword

In [ ]:
# Dal libro Cap. "Il Valore del KG in Azienda" — Ricerca Semantica
# Query: trovare chi lavora in aziende Tech e conosce qualcuno

results = run_cypher("""
MATCH (person:Person)-[:KNOWS]->(colleague:Person),
      (person)-[:WORKS_AT]->(company:Company),
      (colleague)-[:WORKS_AT]->(company)
WHERE company.industry IN ['Software', 'AI']
RETURN person.name AS persona, 
       colleague.name AS collega, 
       company.name AS azienda,
       company.industry AS settore
""")

print('Colleghi nella stessa azienda tech:')
for r in results:
    print(f"  {r['persona']} <-> {r['collega']} @ {r['azienda']} ({r['settore']})")

## 1.3 — Navigazione Relazionale: connessioni a N gradi

In [ ]:
# Dal libro — Navigazione Relazionale
# Chi nel team conosce qualcuno in FinServ? (1-3 hops)

results = run_cypher("""
MATCH path = (me:Person {name: 'Alice'})
  -[:KNOWS*1..3]-(contact:Person)
  -[:WORKS_AT]->(target:Company {name: 'FinServ'})
RETURN contact.name AS contatto, 
       contact.email AS email,
       length(path) AS gradi_separazione,
       [r IN relationships(path) | type(r)] AS tipi_relazione
ORDER BY gradi_separazione ASC
""")

print('Connessioni di Alice verso FinServ:')
for r in results:
    print(f"  {r['contatto']} ({r['email']}) — {r['gradi_separazione']} gradi — via {r['tipi_relazione']}")

## 1.4 — Contesto per AI/LLM: query KG per RAG

In [ ]:
# Dal libro — Contesto per AI e LLM
# Domanda utente: 'Qual e' la garanzia per SlimWash Pro?'

results = run_cypher("""
MATCH (p:Product {name: 'SlimWash Pro'})
      -[:HAS_WARRANTY]->(w:WarrantyPolicy)
OPTIONAL MATCH (p)-[:COMPATIBLE_WITH]->(compat:Product)
RETURN p.name AS prodotto, 
       p.price AS prezzo,
       w.duration AS garanzia, 
       w.coverage AS copertura,
       collect(compat.name) AS prodotti_compatibili
""")

print('Context per LLM (RAG):')
for r in results:
    print(f"  Prodotto: {r['prodotto']} (EUR {r['prezzo']})")
    print(f"  Garanzia: {r['garanzia']} — {r['copertura']}")
    print(f"  Compatibili: {r['prodotti_compatibili']}")

## 1.5 — Analisi hub: trovare le persone piu' connesse

In [ ]:
# Dal libro — Analisi di influenza

results = run_cypher("""
MATCH (person:Person)-[:KNOWS]-(connection)
RETURN person.name AS persona, 
       count(connection) AS connessioni
ORDER BY connessioni DESC
LIMIT 10
""")

print('Hub del network (persone piu connesse):')
for r in results:
    bar = '█' * r['connessioni']
    print(f"  {r['persona']:10s} {bar} ({r['connessioni']})")

## 1.6 — Skill matching: trovare esperti per tecnologia

In [ ]:
# Query avanzata: trova chi ha competenze in Python E conosce qualcuno in NewTech

results = run_cypher("""
MATCH (person:Person)-[:SKILLED_IN]->(tech:Technology {name: 'Python'}),
      (person)-[:KNOWS*1..2]-(contact:Person)-[:WORKS_AT]->(target:Company {name: 'NewTech'})
RETURN DISTINCT person.name AS esperto,
       contact.name AS contatto_in_newtech,
       collect(DISTINCT tech.name) AS skills
""")

print('Esperti Python con connessioni in NewTech:')
for r in results:
    print(f"  {r['esperto']} -> {r['contatto_in_newtech']} (skills: {r['skills']})")

## Cleanup (opzionale)

In [ ]:
# Rimuovi tutti i dati di test
# run_cypher('MATCH (n) DETACH DELETE n')
# print('Grafo svuotato')

driver.close()
print('Connessione chiusa.')